# Classificador de avaliações em português com BERTimbau

Fine-tuning do [BERTimbau](https://huggingface.co/neuralmind/bert-base-portuguese-cased) para classificar avaliações de e-commerce como **positivas** ou **negativas**, usando o dataset público [Olist — Brazilian E-Commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) do Kaggle.

🔗 [Demo no navegador](https://huggingface.co/spaces/Vwhtarj/classificador-avaliacoes) · [Modelo no Hugging Face](https://huggingface.co/Vwhtarj/bertimbau-avaliacoes-olist) · [Código no GitHub](https://github.com/Vwhatrj/classificador-avaliacoes)

**Roteiro**
1. Preparar o ambiente (GPU, bibliotecas, chaves)
2. Baixar e explorar os dados
3. Criar os rótulos e dividir em treino/validação/teste
4. Baseline clássico (TF-IDF + Regressão Logística) — para saber se o BERT vale a pena
5. Fine-tuning do BERTimbau
6. Avaliação no conjunto de teste + análise de erros
7. Publicação no Hugging Face Hub com model card e versão ONNX (só com `HF_TOKEN`)

> **Colab:** `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4`. Cadastre nos **Secrets** (ícone de chave): `KAGGLE_USERNAME`, `KAGGLE_KEY` e `HF_TOKEN` (permissão de escrita).
>
> **Kaggle:** `Settings → Accelerator → GPU T4 x2` e `Internet → On`; adicione o dataset *Brazilian E-Commerce Public Dataset by Olist* em `Add Input`. Sem `HF_TOKEN`, o notebook treina e avalia mas **não** publica nada.

## 1. Ambiente

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U transformers datasets evaluate accelerate kagglehub huggingface_hub scikit-learn onnx onnxruntime

In [ ]:
import os

# Configurações do experimento — mude aqui, não no meio do notebook
MODELO_BASE = "neuralmind/bert-base-portuguese-cased"
USUARIO_HUB = "Vwhtarj"                   # seu usuário no Hugging Face
REPO_HUB = "bertimbau-avaliacoes-olist"   # nome do modelo no Hugging Face
REPO_GITHUB = "https://github.com/Vwhatrj/classificador-avaliacoes"
SEED = 42
MAX_LEN = 128
EPOCAS = 2
LR = 2e-5
BATCH = 32


def ler_secret(nome):
    # Colab, Kaggle ou variável de ambiente, nessa ordem
    try:
        from google.colab import userdata
        return userdata.get(nome)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(nome)
    except Exception:
        return os.environ.get(nome)


for nome in ("KAGGLE_USERNAME", "KAGGLE_KEY", "HF_TOKEN"):
    valor = ler_secret(nome)
    if valor:
        os.environ[nome] = valor

# Sem token do Hugging Face o notebook treina e avalia, mas não publica (é o caso da versão no Kaggle)
PUBLICAR = bool(os.environ.get("HF_TOKEN"))
print("Publicar no Hugging Face:", PUBLICAR)

## 2. Dados: avaliações da Olist

In [ ]:
import kagglehub
import pandas as pd

pasta = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
reviews = pd.read_csv(f"{pasta}/olist_order_reviews_dataset.csv")
print(reviews.shape)
reviews[["review_score", "review_comment_title", "review_comment_message"]].head()

In [ ]:
# Nem toda avaliação tem texto: muita gente só dá a nota
tem_texto = reviews["review_comment_message"].notna()
print(f"Avaliações com texto: {tem_texto.sum()} de {len(reviews)} ({tem_texto.mean():.0%})")
reviews.loc[tem_texto, "review_score"].value_counts().sort_index()

## 3. Rótulos e divisão

- Nota **1–2 → negativo (0)**, nota **4–5 → positivo (1)**.
- Nota **3 fica de fora**: é ambígua ("chegou, mas atrasou") e confundiria um classificador binário.
- Título + mensagem viram um só texto. Textos duplicados (ex.: "ok", "recomendo") são removidos para não vazar entre treino e teste.
- Divisão estratificada 80/10/10. O **teste** só é usado uma vez, no final.

In [ ]:
from sklearn.model_selection import train_test_split

df = reviews[tem_texto & (reviews["review_score"] != 3)].copy()
df["texto"] = (
    df["review_comment_title"].fillna("") + ". " + df["review_comment_message"]
).str.strip(". ").str.replace(r"\s+", " ", regex=True).str.strip()
df["label"] = (df["review_score"] >= 4).astype(int)
# A nota original fica junto só para a análise de erros; o modelo não a vê
df = df[df["texto"].str.len() >= 3].drop_duplicates("texto")[["texto", "label", "review_score"]].reset_index(drop=True)

treino, resto = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
valid, teste = train_test_split(resto, test_size=0.5, stratify=resto["label"], random_state=SEED)

print(f"Total: {len(df)} | treino {len(treino)} | validação {len(valid)} | teste {len(teste)}")
print("Proporção de positivos:", round(df["label"].mean(), 3))

In [ ]:
# Quantos tokens tem um texto típico? Define o MAX_LEN sem cortar quase nada.
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
tamanhos = treino["texto"].sample(3000, random_state=SEED).map(lambda t: len(tokenizer(t)["input_ids"]))
print(tamanhos.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
print(f"Cobertos por MAX_LEN={MAX_LEN}: {(tamanhos <= MAX_LEN).mean():.1%}")

## 4. Baseline: TF-IDF + Regressão Logística

Um modelo simples que treina em segundos. Se o BERT não ganhar dele com folga, não vale o custo.
Como as classes são desbalanceadas, a métrica principal é o **F1 macro** (média do F1 das duas classes) — a acurácia sozinha esconderia um modelo que erra muito os negativos.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline

baseline = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    LogisticRegression(max_iter=1000, class_weight="balanced"),
)
baseline.fit(treino["texto"], treino["label"])
pred_base = baseline.predict(teste["texto"])
metricas_baseline = {
    "accuracy": accuracy_score(teste["label"], pred_base),
    "f1_macro": f1_score(teste["label"], pred_base, average="macro"),
}
metricas_baseline

## 5. Fine-tuning do BERTimbau

In [ ]:
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(SEED)
id2label = {0: "negativo", 1: "positivo"}
label2id = {v: k for k, v in id2label.items()}

dados = DatasetDict({
    nome: Dataset.from_pandas(parte, preserve_index=False)
    for nome, parte in (("treino", treino), ("valid", valid), ("teste", teste))
})
dados = dados.map(lambda lote: tokenizer(lote["texto"], truncation=True, max_length=MAX_LEN), batched=True)

modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE, num_labels=2, id2label=id2label, label2id=label2id
)


def calcular_metricas(saida):
    logits, rotulos = saida
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(rotulos, preds),
        "f1_macro": f1_score(rotulos, preds, average="macro"),
    }

In [ ]:
args = TrainingArguments(
    output_dir="saida",
    num_train_epochs=EPOCAS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH * 2,
    weight_decay=0.01,
    warmup_steps=int(0.1 * EPOCAS * len(treino) / BATCH),  # 10% dos passos de aquecimento
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=modelo,
    args=args,
    train_dataset=dados["treino"],
    eval_dataset=dados["valid"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=calcular_metricas,
)
trainer.train()

## 6. Avaliação no teste (uma única vez)

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

saida_teste = trainer.predict(dados["teste"])
pred_bert = saida_teste.predictions.argmax(-1)
metricas_bert = {
    "accuracy": accuracy_score(teste["label"], pred_bert),
    "f1_macro": f1_score(teste["label"], pred_bert, average="macro"),
}

comparacao = pd.DataFrame({"TF-IDF + LogReg": metricas_baseline, "BERTimbau": metricas_bert}).T.round(4)
display(comparacao)
print(classification_report(teste["label"], pred_bert, target_names=["negativo", "positivo"], digits=4))
matriz = ConfusionMatrixDisplay.from_predictions(
    teste["label"], pred_bert, display_labels=["negativo", "positivo"], cmap="Greens", colorbar=False
)
matriz.ax_.set(title="BERTimbau — conjunto de teste", xlabel="Previsto", ylabel="Real")
matriz.figure_.savefig("matriz_confusao.png", dpi=150, bbox_inches="tight")

In [ ]:
# Onde o modelo erra? Ler os erros ensina mais que qualquer métrica.
# Os mais interessantes são os erros em que ele estava MAIS confiante.
probs = torch.softmax(torch.tensor(saida_teste.predictions), dim=-1)[:, 1].numpy()
erros = teste.assign(prob_positivo=probs.round(3), previsto=pred_bert)
erros = erros[erros["label"] != erros["previsto"]].sort_values("prob_positivo")
erros.to_csv("erros.csv", index=False)

pd.set_option("display.max_colwidth", 200)
print(f"{len(erros)} erros em {len(teste)} exemplos")
print("\nNota alta, mas o modelo disse NEGATIVO com confiança:")
display(erros[erros["label"] == 1].head(8))
print("\nNota baixa, mas o modelo disse POSITIVO com confiança:")
display(erros[erros["label"] == 0].tail(8))

In [ ]:
from transformers import pipeline

classificador = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)
exemplos = [
    "Produto chegou antes do prazo e muito bem embalado. Recomendo!",
    "Comprei faz um mês e até agora nada. Péssimo.",
    "Veio com defeito, mas a loja trocou rapidinho.",
    "Não é ruim.",
]
for texto, resultado in zip(exemplos, classificador(exemplos)):
    print(f"{resultado['label']:>8} ({resultado['score']:.2f})  {texto}")

## 7. Publicação no Hugging Face Hub

Envia o modelo, o tokenizer, um **model card** (README) com dados, métricas reais deste treino, limitações e exemplo de uso, e a versão **ONNX** usada pela demo no navegador.
Sem `HF_TOKEN` (ex.: na versão do Kaggle), estas células só salvam os resultados localmente.

In [ ]:
import json
from huggingface_hub import HfApi, login

repo_id = f"{USUARIO_HUB}/{REPO_HUB}"
if PUBLICAR:
    login(token=os.environ["HF_TOKEN"])
    trainer.model.push_to_hub(repo_id, commit_message="Modelo treinado")
    tokenizer.push_to_hub(repo_id, commit_message="Tokenizer")
else:
    print(f"Sem HF_TOKEN: nada publicado. O modelo de referência continua em https://huggingface.co/{repo_id}")

resultados = {
    "dataset": "olistbr/brazilian-ecommerce",
    "tamanhos": {"treino": len(treino), "validacao": len(valid), "teste": len(teste)},
    "baseline_tfidf_logreg": metricas_baseline,
    "bertimbau": metricas_bert,
    "hiperparametros": {"max_len": MAX_LEN, "epocas": EPOCAS, "lr": LR, "batch": BATCH, "seed": SEED},
}
with open("resultados.json", "w") as f:
    json.dump(resultados, f, indent=2)
print(json.dumps(resultados, indent=2))

In [ ]:
card = f"""---
language: pt
license: mit
library_name: transformers
pipeline_tag: text-classification
base_model: {MODELO_BASE}
tags: [sentiment-analysis, portuguese, bertimbau, reviews]
metrics: [accuracy, f1]
model-index:
- name: {REPO_HUB}
  results:
  - task: {{type: text-classification, name: Classificação de sentimento}}
    dataset: {{name: Olist Brazilian E-Commerce (reviews), type: olistbr/brazilian-ecommerce}}
    metrics:
    - {{type: accuracy, value: {metricas_bert['accuracy']:.4f}}}
    - {{type: f1, name: F1 macro, value: {metricas_bert['f1_macro']:.4f}}}
---

# BERTimbau — avaliações positivas/negativas (pt-BR)

Fine-tuning de [`{MODELO_BASE}`](https://huggingface.co/{MODELO_BASE}) para dizer se uma avaliação de compra em português é **positiva** ou **negativa**.

[Demo no navegador](https://huggingface.co/spaces/{USUARIO_HUB}/classificador-avaliacoes) · [Código, notebook e análise de erros]({REPO_GITHUB})

## Como usar

```python
from transformers import pipeline

clf = pipeline("text-classification", model="{repo_id}")
clf("Chegou rápido e funciona perfeitamente!")
# [{{'label': 'positivo', 'score': 0.99...}}]
```

## Dados

Avaliações com texto do dataset público [Olist Brazilian E-Commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) (Kaggle).

- Nota 1–2 → `negativo`; nota 4–5 → `positivo`; nota 3 descartada por ser ambígua.
- Título e mensagem concatenados; textos duplicados removidos.
- Divisão estratificada: {len(treino)} treino / {len(valid)} validação / {len(teste)} teste (seed {SEED}).
- Proporção de positivos: {df['label'].mean():.1%}.

## Resultados (conjunto de teste)

| Modelo | Acurácia | F1 macro |
|---|---|---|
| TF-IDF + Regressão Logística (baseline) | {metricas_baseline['accuracy']:.4f} | {metricas_baseline['f1_macro']:.4f} |
| **Este modelo** | **{metricas_bert['accuracy']:.4f}** | **{metricas_bert['f1_macro']:.4f}** |

F1 macro é a métrica principal porque as classes são desbalanceadas.

## Treino

{EPOCAS} épocas, learning rate {LR}, batch {BATCH}, `max_length` {MAX_LEN}, fp16, GPU T4 (Google Colab gratuito). Melhor checkpoint escolhido pelo F1 macro na validação.

## Limitações

- Treinado só com avaliações de **e-commerce**; em outros domínios (filmes, restaurantes, política) o desempenho deve cair.
- Binário: não existe classe "neutro". Textos mistos ("veio com defeito, mas trocaram rápido") recebem uma das duas.
- Ironia e sarcasmo continuam difíceis.
- Notas dos clientes são um rótulo ruidoso: há quem dê nota 5 reclamando, e vice-versa.
"""

if PUBLICAR:
    HfApi().upload_file(
        path_or_fileobj=card.encode("utf-8"),
        path_in_repo="README.md",
        repo_id=repo_id,
        commit_message="Model card",
    )
    print(f"Publicado: https://huggingface.co/{repo_id}")

In [ ]:
# Versão ONNX (8 bits) para a demo com Transformers.js rodar no navegador.
# Refeita a cada publicação para não ficar defasada em relação aos pesos.
if PUBLICAR:
    import onnxruntime as ort
    from onnxruntime.quantization import QuantType, quantize_dynamic

    class SoLogits(torch.nn.Module):
        def __init__(self, m):
            super().__init__()
            self.m = m

        def forward(self, input_ids, attention_mask, token_type_ids):
            return self.m(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids).logits

    modelo_cpu = AutoModelForSequenceClassification.from_pretrained(repo_id).eval()
    os.makedirs("onnx", exist_ok=True)
    entrada = tokenizer("Chegou rápido e bem embalado.", return_tensors="pt")
    nomes = ["input_ids", "attention_mask", "token_type_ids"]
    eixos = {nome: {0: "batch", 1: "sequencia"} for nome in nomes}
    eixos["logits"] = {0: "batch"}
    torch.onnx.export(
        SoLogits(modelo_cpu), tuple(entrada[n] for n in nomes), "onnx/model.onnx",
        input_names=nomes, output_names=["logits"], dynamic_axes=eixos, opset_version=14, dynamo=False,
    )
    quantize_dynamic("onnx/model.onnx", "onnx/model_quantized.onnx", weight_type=QuantType.QUInt8)

    with torch.no_grad():
        esperado = torch.softmax(modelo_cpu(**entrada).logits, -1).numpy()
    for arquivo in ("onnx/model.onnx", "onnx/model_quantized.onnx"):
        logits = ort.InferenceSession(arquivo).run(None, {n: entrada[n].numpy() for n in nomes})[0]
        obtido = np.exp(logits) / np.exp(logits).sum(-1, keepdims=True)
        print(f"{arquivo}: diferença máx. para o PyTorch {abs(obtido - esperado).max():.4f}")

    HfApi().upload_folder(folder_path="onnx", path_in_repo="onnx", repo_id=repo_id, commit_message="Versão ONNX")

In [ ]:
# Prova final: carregar do Hub, como qualquer pessoa faria
publicado = pipeline("text-classification", model=repo_id)
publicado(["Amei, chegou certinho!", "Nunca mais compro nessa loja."])